In [ ]:
# Frozen V9 on 1,000 unseen synthetic notices.

!pip -q install sentence-transformers==5.7.0 transformers==5.22.1 sentencepiece
import re, pandas as pd, numpy as np, torch
import base64, zlib, json, csv, io, hashlib, time, platform, importlib.metadata, types, sys
from pathlib import Path
from datetime import datetime, timezone
from IPython.display import display, HTML
from collections import Counter
torch.set_num_threads(2)
torch.manual_seed(8)
from sentence_transformers import SentenceTransformer, util
from transformers import AutoTokenizer, AutoModelForSequenceClassification

device = "cuda" if torch.cuda.is_available() else "cpu"
labse = SentenceTransformer("sentence-transformers/LaBSE", device=device)

NLI_NAME = "MoritzLaurer/mDeBERTa-v3-base-xnli-multilingual-nli-2mil7"
nli_tok = AutoTokenizer.from_pretrained(NLI_NAME)
nli_model = AutoModelForSequenceClassification.from_pretrained(NLI_NAME).to(device).eval()
# find which output position means "contradiction" from the model's own label list
CONTRA = [i for i, name in nli_model.config.id2label.items() if "contra" in name.lower()][0]
print("NLI labels:", nli_model.config.id2label)

def similarity(zh, en):
    a, b = labse.encode([zh, en], convert_to_tensor=True, normalize_embeddings=True)
    return float(util.cos_sim(a, b))

def contradiction(zh, en):
    # probability that the two sentences contradict each other (higher of the two directions)
    scores = []
    for premise, hypothesis in [(zh, en), (en, zh)]:
        inputs = nli_tok(premise, hypothesis, truncation=True, max_length=256, return_tensors="pt").to(device)
        with torch.no_grad():
            probs = torch.softmax(nli_model(**inputs).logits[0], -1)
        scores.append(float(probs[CONTRA]))
    return max(scores)
frozen=json.loads(zlib.decompress(base64.b64decode('...')))
raw=zlib.decompress(base64.b64decode('...'))
assert hashlib.sha256(raw).hexdigest()=='e99de209ce48848ea2d4ff3a1a68adcb421600265a0043926be9bdc673fa13e9'
references=list(csv.DictReader(io.StringIO(raw.decode('utf-8-sig'))))
assert len(references)==len({r['case_id'] for r in references})==1000
# Only bilingual input and identifier pass to the prediction loop.
inputs=[{k:r[k] for k in ('case_id','chinese','english')} for r in references]
patch=zlib.decompress(base64.b64decode('eNq9fGt31MaW6Hf/CoUz96p1LHqFzJp7swwNw0rIJJMTkpWQ3HWP8XipW2W3BrXUI6ltbMtrmYcBA46ZE8cEcAIkPJMQ3sTBhPyXc61+fJq/MPtRenW3DcnJueeBW6WqXXvv2s+qXdq2bdsnrytlt+GYwtSVyapri+2OG1gVofiiZjjwSxn3jFrN8IoDAx8Yni98RRiVqmIbznjDGBeKBWPrAv5xAnuqqOx3lbJwKlUYcUh5501fh55lYcNfcdio1W0Y73oDnhFYrmPgk+EJpeI6fsMOhFlU9k0Ib0qpG0EgPEepNfwA3kKbElSFAnNY0J0x3Kk0nEOOO+kMVGyj4RNgghR4jQqC95Uxw7aVslE5pAQuARjz3GnhKJWqqBwSXlE5ULV8Bf5nKGOWYwViQBIvadZxKmVcOMIzbCXwDMe3CXMFOeb5geGYljOOE08Ka7waIBixPWg40Foc2LZt24BVq7teoHgi/gXvKi6w2wiMAUCnpuAvoMBHCmSfpGlg4J/T3/QvMBgxHBpQ4D+HgPlDChBMT4Fbtyrp45hRCfwhxbQqAb8WhwN+O/AH5QD2ZeY7sPxKveoZgAATbHhlC4iFdfCB47CawoeFOQxNSsxqWhjsWxYAzZ8ERruTwlQM5CUhogBvkOeeKE8pYsIwBa2A5UATcNpMFmFg3wGlpHhqYc8Hu2jk7uF/K+o7h17Zc9AZGdyjqQN/7u2g/7+5o9ku772//8Dbo/v3vrfvI+ir/qvhNBD/t0TZox/vGR4I7d66Z9nwe0r514Yj4B97StnbGEdaPhL1QNTKIGjvVwIX/+4HsaOGN0WFfqhFv25bQUGTs7239wOYa8YvIuleQRtSrMEdyhgIg6XDUjogro0aSE4gChn0tFk5/IO9Bw7s+3A/ohuqxX93LSfXbWDAFGPAYq9WwJXTeMU9ETQ8JytERexi2Na0KKj733r3DVWn7kVP1G2jAo3/9eysqsN/c00XoWEImkAcrHpBTlax3cqhQlVXqoY9pis1y2kEorTfdYScvVqynKBQ1XbG7/CRf6IOvKpRL2uMJGOHsqukVPGfHa/hW2x7FR/lgF3K/3p1SPEMyxfKJ4bdEPs8z/UKquVMADkmY6NqWbLH1Jnq/9jx2mABIMIsiGbMfOR3YVt9m75tY+1MtHgKfjQvXNxYO00td2TLnW/wh6aAQRKA7tCrr5mzQzOMED2okhNBoe6JMeuwToZOkg9T4lOppApH5aYMct4YSegMD5yt7j5ozszs0F+bndUKe4aGsu9gQnqL77Q9B/0/5gYCVbuHjfqIdrC4pwb/Vwe2mAU7A5VAVsiEh0x2GJ0827xwK2TaQ6Zc2wzB4eaFo531r0YA0X54cqfo1Altj2RQYNVAp+2CqddLqpoXThYjc7g+qFbVEZ1/AJ74uzgugLWDKkBWNSl2IMUCDH6hrm/K6x7qZ80MYgf9Qdla2z2TU65ZrT/zoGM6vLl6qg/Q5vnraoqfpDWLYa2UmIFhorGmjsTSOJLHn+UNlSXuyGJtGlOlpNWMW2P92VWq7SqlugPP0H9X6R93bKU1iGyP0tRItrfPwPhYyj/8+E9kKmeIv8MjuqJOV/HHLNFsmGYB8dfRxejSHevgXcEVx7aIQAxjr5GiUccYoFCg7p4oVtxa3bJFQY4c9NRhsNjFkT0qvn1HS0Bp6I3e98Cboh8GC+UJcjrkSoJJV3GMGj6wRwf/JOJgoO7aBniSKcUdU8BEgdc5JJziANpfxAkNAmG4pe5ib4atu+isiF4cOlxQARl0xjJ0AAm2wWhWXNsWMAAmmHIbHkQX5riAdvoLSwWEqJpeUP1GuWYF3McT/9EQfoCjPWHQWMOBaSvgebFV/opHj6ToEYoWBBq6L4BjZqkQI5BMnGJJGGpIKRIBlGZxYhEsMAYiM7lENMZRy82dFwNQn7dcZv7MvgOzOhiuGWberIIaRDPtNsZgxcOyANYKTZkhts6qeg4u/sc2amXTUEzdKDGJ5RITOTSjMlSJ+hBIsDGrMERlpjyrIo3msMqvE6UDimluSSuMKqejjFl1lmnDl3kBIOIMvSzXPcs2Xe1c+SJaWonOLbYW0I9Gp+61bvwQXTkaXbkPv2mt8+sHQ1bOtL47E609bJ56Ao/NpXMbT6+1lh+2vzvTs7oo3n5pxhjqWtgtl7FnFctD3XKw9fiuRZ/detHBK0Qnn0Zrd8Jo6Vj71Mmw/fxG69wJbebPB2bhXXN1MTpzQdsDPgRFAqiM5smiGruB7SGsgobt0YlFbCwnjVvJhFMixvSIggOmVjXUkZFBNV5adZAay9A4m0pvQkR+OYEawI/XFNHndY2en20uLHYvrR5j04NFfq1iRLpWoC8yXaIi0cnIC6PCItPFXylBW6GVE4sYr+7FJpP7IQS1Vj1QyGuAFR1Saq4TVMGYjvkioOCd8kIIniExtOBNc3UOkNtYmwPPCLkQziTjfU4eigNIJppXNEoMXRyuW94U0blXiRvHMjYEjT47LkAYYnPO8ky06fiTcSJZgpzS9XaDhk5WrUo1JE0HYxcokwamcH5DmDnWdKEwpEqw1JUBx3aEYecMSTxPrDBy8JiLeQ4acAYAvCSiwXX2JRpXsHXxeHP5SfOzi83Vhebnp9onH7YuPAN9ylDV+vwJBCHwv+jp5yEx+QzwOZo7A004nF5ofyt5QFZ2ot+BNKaLMETSmquXmyvPNqGumy4tK0/R/RPN9V9+NwJfTBpowH6YwPdBvrej07FIkSZBk6CfTIPBR0NW7rGKjYEejY1ZFQuiE6UuvBogAi+KA+JwBXJH+Dk6gfFYKUODb5liu+mCvCedgBA/MMbGFA50KgI1wIlRId1wIZQCJGLqELtUNinPURuODQPQ1Vy41vnyK/RLS99Gq7dULXYQtuuDekKw5LD2ZidFg48T1cnhwVq+2fCQO1nHjr8RfYXQBxwlRFxXxGg3oxDyDFXhaGAIeJIap80wBvhmNiBCo20AkpoDObDdluBvn0N619QSdXEfQviu9dLyrCCxXr3cWflMl3kZYcJ8DqPFhxs//wKiu9A+8mP02VVgeXTkUWdlDuQumnumo6c8dyNaOBlurC9q/KpzfrWzspA1B9z+ErDbtx+0L3zeF3YXVHITMdYhzKF1TcLykUIHRc1Cx/xv5Uxz+XmOg6T7L8FB0KZ/8dxGXXHLtjXOO2SwTlPgHILAqFRhTWUUPY7ddoLNtg9tB46neoRL6UMaiNslWU9CA8AVuYdQJWIRIoEBNwdo+DEw/2ARaGc8JizfgjdIfc01DXs37VeVBQESFOAb5gTYDRGaLim7I9CbcVs8Xd7iQjBkTVhmw7CTGUGZ5cZXgmkXCBUDs/9oWB50kfpMCKUKXQRGeYGPSlpQEc1EhxGtZPBsPzn99Hq0dB7kJrryCAKW9q1v9Y2fT0SrX3S+W+bnlAGdK6fC5l2IfE5pnSvnWo+OEy/BFEenrm88fRp98120dDcLCieAlE1CWzjZF9TGU4z1JMAeyelau/rfkZtg+gEfdRPOYbzj2gLNuTXu1NCC14y6LzM/Cnbc8r9jQAdiCo2up8P4ccMz0f6g7ed9Tw60igN73zjw/ocfje7bX1J5UzbAPXHh+SFunAAcP0T3VY8VYWwsnHBtiHAE9FHj4X9+G/zV0p2NtfWN9fUQYtL21yut5cshLuvzeVbTsPX8dvTjdTkG9xpKM+kgDPly83O+wnDgZYwNJiEZoMTGHHrQQU40pGYwnc0oIhhZS/iqzAcgDwz3BlrOZZCtwU3d3TMJg2Y13ueljrblc2SJPU0L9x7LjUDku6ftHAGCPfAsw/ZzukiQCDAl07hi6hD6SmxKdUtXk+EpXFrxZEQ6XTqsn66lBpasay+tf357Vms/uNJ+8H3zylp0bxEyi2jlB30TUjO9Hz2HiKm1cLv98GQaWNFU6CpOnYBViW4chdBpiEG2lu/3n53nBZ1lYLLfJpPztGwEVm/hnPpLgdd/BfBeVyJlqP7CtUzEfThd1ZEXLWduUHZhMTHMiDI4RPBKgdKo025Zv32NKbcBojclu/A2TjzMaVDskbhWNN1uA2mAFA9oCF3HnlIoQUnaWFMwp8WAhcAlyXuGF9SBxgiM1ciGgScToJY5m9Y3NMx5EolU4kzioXFSWK/inpnkQC7W78MdzvdJMCHHB0fRvPSouXKvfeRu6/HVzqX19u37esIOdg3YY/mnEPwRPICP4GdNJrk3PwNh+x0oz1LM8/4+BGfRBHI3fvmy9fmFjbXF9g+ndYi+0FVCdJblSi9HfhN57KzecD1PyDPMMm5IVhrQ4LDx9BsQNUGYDKRR+IVHTT44AEp1cN/VoGCe3kFIFfi5xNx1a9ApBs8GB4MpJn4IRB8Uh30f+EzoTBt7Nu6MD2rwMAQ2kZJ0w7MtFHDLqzRswwOhz7Wjo9UoIvsYViOBJAkhaDkGxRQiguoQqG6BN8bjF+qIpuFmZUJ7T0dAEjttarznYH2iHxaan/4ljFbvdda/4iXThtrfnYHAu7V8qxvB6IdrGHU3z9+OFhY7cxej/zwdtk+dhj+ta0+15sJcdOo82Ogsh2AIxuk0jbYn+vEBPi2tNJ/g8cxi58htmEbL5M8/gRz1mbZ9aqF99SyYcpiOJ95sNpgiAZzmGDfX2ycf9oGLEL87E535ZeP5l635mwkxQF/0fK4LdK/97pGe+t9zCfvs4gMkHTqV4iMjBK3yoYyetCEUbttq51+eTPilYRjBGfdLKwYdP3hiwnIbfpyu08kLquAMzD9Lko+ZCvbCnJbeQo4zAxjP8q5xve4C3RSNdk2c6n8uRR5ixwSuIzOUU2ACS5N+YAsDNG4TZEXNsGzFr/J2CKGqaiN99sVfhj0gaZw0Ns9fh4gFRBEEunX0KSGjR59ejn64SFNgIEMqAV1YG4gDW8CN7p2EDJUA8RQguayH0fz30doxgEiQc+L/Aq7K7JuzFUY5Rpb1YmMd2r4H69059hjeNJ+u6Dxnony/ek5pOwhkDxRmBcy98fMVbEZ+MnLLP8XUx6uTnFOQYYtXZ2ig/6Z9qpe6XcLmjHqSkxtKDzWlDuVVs6cTK5XG/ulfZEUMuqNkMwCfG6ZFe1p4PIyneSBtTpLWQ1B0SEwJE/Knj9/E5GlG/UB4QFgNcw3IVtIHXd13uGqVLQjr8I1IH3T1HWeC6kf8ujAO8Uiru0lXaR/A570vmT1iT5n/U3aJ+Q1iAnnYjNq+eqv5bLk9N9+NSLR0LLr/Ob/I4dFafNA5Otf+/hK/64MDLDGvJyfRfaZPHXPCOh+3Sxq2SC1Q3HLQH4wZT7urJGW7Dc+zYI1Cw5Q7KRqaiZlKUNgmC5e26duEs02bhfEH/yjztMmquztlfphyO+xmb9iPkxphMCG88m4+9QgJDRHi0a2nKUagEAYApGKlGMSRSs7zx+VVTA9uAsSUJPu7/CZ30Jf24XBPsgF3DuJyLZBBdSgtVIhfgMqqidAmIqsOOeJwUJhgRdMnUMtYTIvAjppf4IPVFAXAC5iYIpWHKmnB0/WAjmZShOMjWzU+s8PVJ+5hN0nGLKYwyN9MTUF2gh7iJKfVvvEP7e59BiIOIrCxNtc+9X1GgKJT96Irj7hLGM1fhwdtqEt+pqt95GdmW7iN65ZYi2Dq2HDllp4GJ/LCh2shzxrPl9jnTbCL+zGWZLV78QNDugVqmJOcXehcON6+tf7bcOwJh/robP23STZINM/y+4kzEz4ci+lIf/GUs+blU55/ZuSTOdFfPl9eLsFzfGRhDSh1w50n0CxQEH8oToAUgzIYWwQiqT3kwxgwU7QZVKlSrGWNW5jYpF5RFozyCT6lhYyL3K/CUEzbw+aUcsbERMW2KReRJEOTzWZ8s8mgrPtPBv7f7jit+6yjF1T32UWu9CYjVl08jjU+293vw4jEGEjhj6Ue4qvo5hnWwN52HJGN9qCJlXAo1xnC9h72bTEhHjT0TNh+/CPm2pSK0bQj/bmLcHsippxhSLDSuWviiLtRSVN4mEzyP7GeSZCVqnxXcNV/MaiAqo610qNcTkwFnNnSMb+U1nV2RXf5Sqsk/eEyqzTcq5Vkz+JYw7ZJLwp+WnAgS8hqQ7RpZTkNkbwyS7UixR9YGFzQdnK1Lroz+kGbKlwTmgEmEyhgAcV8srCk+2QzV86R3afB3Xk+qKThKp9jqYQjBoh90OwDgg/IJIiN9cUtx2OFA+3o40GhadKZT6YynEvF+dTXV6oW9EDjQUeTBshZAEbKt92g2IcLWNCHOHii6AusKy546sEyGJmK4YQ1YyrEs6RwEs8j4V83DKpGEOLJTijPM3F/MITOgH4o9+lkvRSXNYSQnzVsE/4Yth1irzJu6YR1gM3FHCHkdpaPXVEKHDOse+4E4G6GvByh5VTsBh5lawfLqk4ry4V3fVjVu75ZyvC0p7M6F4JqhvFJpTyGXFuMnp+nPydPhNGx+c6Fc9DtW/pDaxXyZgN0uQH/Xw6by6vR2pOw9dPDjfUvQi5yCdHbnFsMN346Ez0/S38WFiXK/bANvKkhrm0vSSUBrUveMvlKIS2N1N8VU/RjE9Ljgr/uuq+uojziC5evqoaK8R//LstX0IhQzD6TyDpMLtznCkleEKJCT61A0tERbERYYBueMEflxp7vOgV0gcIXunDGbcuvSpsyrYtSzujE3XBVNT33So7UUZRzNafTcbEp1XNPFxHZV0qiyBcNsvgRs1HkwEOXhjM5qpgilYS4QZgFH1g0XSQ6tRAfhHzQUmMxPaGLiZLsRUwFGDpti1LwMGYJEwgQL+iQXdPpCUB6Im+PYmTjUlUyfxWIQMZdb6qEECXDZBkHosVskg1iIunh1w0HMCazHneiNkFtWm45YWH0ePIBvBPyDhgivJjyyesK3WwxlDoGVUFyPYVK3+oGhj62MCb4hglEPJhijzUcacIs2rUu4iUTukdCb1zXTm6R2F5jtILnggOjDFgu/yidp4xOvK6U+NxFtrPMjaL8AhENwx7F+zN4fQb6biJ1YLpKm0DvGTHAdvlA7hYIbjFs9/BwtioaIN9448iwfZdv/VBFsG3hNSHQsQ8/+uQDv6jghrIhgTmiEWB0TJeMFM/AKhAIs9LSHeCdoCtFJuSdNiBEsZeDoZltVahUoRhTgqHyFLia9Cf6wvTaRGZMCs60xsYA0SFVV9R+jWCpxeHSW0CTyIkFTDIw8M/JGhVqxmHfmhalf9rxmsYl9mgjRv0KuAUf2TldBU5Ktv9BeQvcPsiNW3d96cRQ+/b/6R0Ol+PgGUTKGKcYu6i8ZR0Ghn/yv4E/ALLq2qZfzGJU8K2axTXVcjKdjJlnoK7AHAUVt8fmLrYuX28+WYiWTquD09VBFfIusJd0lUpen4JRFYpCMeBVB4UzqBaTcv+syMUyMppTpcx9FzJbZmlLQ9hncGzTeDguORqt1CCQ2G4t6JtBTWTFBNODCdMoRGJVkBpVqkEMS+12ALji0lInBoFNtslRoVXTmd+lnrUvsscQ/DdWpb14GFQnU5v6CsV0hc9GHOxMGdFXMLsCK0YSIUsrsagKoqfAF/ZYLAYGcNWPrTlEGhhAjTZwLQPcyCshirs+eue90QMygAJcd4PMjR6QCO3HC0+gwUhcgIbZo0wkjqioCsdveBPWROaCYZzVFeNVQ0cBIcgYOB3gJUZX5qCqSx5omvJKaZMuMXsy3mVrs69KIjMIq91+IGZ93hnI1m6PkOsauwTGKaGuh69o22i9JK5DmSh2PwQh27nAVmF5kJf2AqVi2FbZozJc2zok0syZdQs0/k2qbcpAiwuEwW5Ogpk1vBp4aht3ZstTuF3rTvB+PCkGjBdgxZSGA7O69gRWceEyZuBl5UpWeInDaMQBCiYm8WVSeddP5wJEj2bBAi0bLE7gF/++epkwejRmfa+K0kipdYkVhMHw0K9PzixCN37eTOHRMh+mC4E9Cyx1LpbOsW0zhyHjkeKpjswmnkTZxxQr2CEni9Brp/IGM4fe5uQX3uKlWp5X+evccgJoOAsJRRW6jqSQhrOg4tfbci6sbhbfNALjLQ/ilcIwaRbu6TT8kvrXi1f+a20JgGHlBkq9JFRub/11dV55/91E1Ur91raUXWMd3HFJ3anIi5ISmqbHRmQ0MajxDz0nB6DpqQthoVR1GbBQNsuxrt4tAigAep81lyueyCEnJDJGTawCt8YB8Ag4wDfe3vfGu/s+HP1k34cfvfP+/pI68fr2FDOpLtsreK+YjmAoYHwDFFu6eQh1yhaEGONYOCeL0nhO3qDP6iqwDVcKBB+eUAY/eb34cleS40tOFGbKK8lizGjYAd0mHnjjT3s//mjfaHyHDBwGXx9FdCg4V4SjKxBDwF9pMNH/YlPmUVrpLLD0HhkAyVwjA3B8bSzTNl2FpECC12PAWuberGH/PW7OXoKGnaqcZgxDKkIWMJyOg+IMI6g53i+qUUUeFWD3aQKQDE7FReT0Hms8LDHJpa+4p0hFSdhEttYT4BFMPpvBCqR4GPQ/WMQRzcvXOyv3W4/mOmePt45fhVy+dWG9fesLJEBiqfIgT1btVtw6bmCqH2ZbsTQbuN6AuBwi6QLVN4FOY5HUpOFQ1aTpTjq2a5i0g1ufwu0PisVlPrlFV41RLUTH5qPVW53VufaNIxtrZ9rPFqKFuxBwYnHNwmII6Le/W2l+cTc6d6O3kx59eVKLfplPKKSgNGVyMifiTgapNrwjXz9Eb+LqIczPkM10kPerwAAZfaFo8epms4gxgYsImZuJLNibfwNMh0SlHPcQJjOKrwG1H6xHS3c7c3PRpa+Q1nQ9Ky7kUcKTo7A4ABEzPWMSjUd8VQnRQmhvyN6VquGNy+CiIA7j7pEwQ8rE4ukhHUgq8Un8spRYfjKNk5+FseYNIcYdq+bv3gbBxKP1v1wDCvSNpzcKeKg+N9c5ejxsrh7jX1pn7nLnyFdAbNd6xhgma4D7cklbXAHGeL9oXDJvz8AMU6uWbXpAN37yQtiQjKMZDvAqDJ0RmMI/pGBkg6sW9y1AZ1MpUOUU6AHogDMuvJC+HcHNmjwWsLgWeQIgSsV9tB6d+bz53bGNnxexdH/pfOvciQKNad550Dy1srF+HeQfOPVtc/2XuF2L5v/S+u566+JxZnSeaQV1V0nS/dqIJHWXqg0W6BnwqA3/44j2a/sncg35JMmRJN5zG1RMp76fb+e9TQUdF9f3Qlwkj5UhXRsXUlx+etT5+Wr72bHo2k2sHiG6kkKtjbV13s/vXDnS+fJrln+JBtU8Q9BaRz1FUB/LmbiF5iARpovE4Fd9q2zLWWGC1qWrnZNL8TSnwcDwHFmLCYMC0CXaiuXLC8leMkL5SPCXL8A7QM4DgTX2V2KjSeU49OEUXyk0HGPCgFADMAiTX9Ig8o0Nnr0A1q/58HNGC7dicfnpQWPziLbg2OPo7vHmysmN9Sdd2kLzZebKSn+umReZuqetLwkoi+ImkJJFqgqbQ5I00og1CU/bwZTgh1twT6innzSCC7eBTLATXJoZPQfLn/NqRgNcokelV8mlcrBUkHemd1TI5oJ7qlkOZVFdHQu4l48pqxZPT9cNMcyahNgwQGsgZzEymC0+aC6dw3ulF49vrH+DdSVnjzQvrcEqNj891fz+dAF30tlVadAB8O5eLbwUA+BzRgpwUbsqW184BibqGZOqayOgC54knbhMdWOKrqSqXLZGl0VRQ/AaDy6BCWYPd9KkdN49Hp18CoYZVru1fJ/tzcbTp1zUno8vOFnongZLrg0ZlumkFhDWV6w6XrALk/k1RoAv/BMEnp89H4sBFnUde0y43C80l59AE2ASJjhpzWVQmC42I4/Spxk1Gce3IxgPLPOJocjLJ4SVOjuM40dSKumaOwmvL0u/6xATWBil4w0W204k7h0OnLiZvlyAo5gDOSjoiQ/haYYsraWTiJCydAzoMVBPmmObceMoF5VF80821u5srM11rszr8NC59gX8al36obCxvih1ZvVW+9bRaOWHaP5mCP4XlCl6/hU3tW6udy5c07oEM04m+gc+iGqm+pqwfPnReJ7XPTiRVTCvYxbE6hRpyPL7BBwHxrQd1dOJL6HRXUbIUaWCfv20/e1i68db7QuXW0/ugHZyYXc+igLwuCPEF2Q8rrw2G56RNVVpH1A/yzXj0gK5XiaE6nxbsUCXWSfAxtfxoE7wbQAsvtqJJsU2fLysRnECDPIlnhy1L9/nSzso2FiU0lz4KTp5QkO7cuUZXnc9f739+EdeUo4Dzl/v9vvyLm3WNkiMknoTQgzrNAYxyx5EFz+oEjYvgsR4vQygjAcFdtG3OyiWhBwbUkvDEW4DLxip/wcby6ALslBO4fIoyB3GKanQOdICrrtYTFJACYoPUjW5uZ/WXeC3JaR2zN9uPT4ub7udO4ue8/TjzpFlvNTAXy1oHX1aiH58gGGVvCbHvfGo8vklvojfpRRc0pdlCGISs6PhyPdbjoApewYkwu9DZARUThoWbauBBWpQXEVfSkPzISMq7KDI6nyUdgVPImQ0uTDXXF0AUvAuBgTf5/Dadefi6Y1ni607C9HchbzsY1YC0PhDK/KoIJZ5fIexPr6m5ILMD92RKxgBnlf5QVjD6B9PYTTl7XcP/gOJpYywyLx0Tv5n5+piAVgftp/Md458qrHkrt2N5o/h5Yr5RbDX7Fx25uRvdynLt3jCmHm7QeqkwL397ptbDISZNxmT+ke8CU5XsoW0y5MQTVBAGz+wzW74uBeDvSgfEn4sbGh0pTmdvxmtrYGiyEQmF6a4Nf6SQgU/ZmcLc5xqohTeSaMQheIM2Q13IqZiW+dLy1IPwljvtHh/X/oqaUserrce3+cyZIgr2QTi7YS8UemXNeT95K8xADEnbby0pcTbcdI7phno3vRVHPb5xIC0bJiHsC3NJpXyovTd283L1/HDGeDBlm60r57lM6rW43PNr1ZzyQEd6YO4WuNUp+Tihf406KF75LJNyXSVnzwke03qdHdpY/1naYEh9qEb1HwjMzp9pXXvZz2zwggANSU+wK1AQOmalkFehKIemUZD1JNFQFMOCVGXzisPQyr1nW+aq99jyJOEKGEWFQ19wg9Xt1zNZOpcIJ9EPHJJs2hl7FLVwP0pHzcpIP0NMIhOgmpqHXfpcpI8GJD9xyhPlCR8eqy5tARURHe+wErEeYgjb7UfnmyuXIiuPeeVo+tQtRonWRV0uXipnG/28bFwrrBHbplUaMsUpqNw3XCmkkONZGFAxkC7x/E8BMJ7t0FisI8ae5JyNHlgCt3GeJUj1XFPCCcs2w0BAZkZTgm8uKwRNEna6vecw2P9+Okr7bl5TC6ffFZoPbnSXngQts8v4p/Wo3n801k/Bn+0zsoX0dI3XJO5c4tYlWFAQEpoQJDK0LDAHLf8IWglsBSx4mVqBg+PjGcavabxFe4a+bmsj3WC8vP0PZf086UX/PgLGT9p8K7djK4cZVOX3NnoVQTes8SPaU4lvh4/vulX8ROkWO8gsjIvu0u/VgAPP+5iLZQFGNX4YiVAgiWn0i2+62Kh4krDd+/n1s8PwDaQg793koMqwA2Y3Fz4LFq817x9uXn+CF6sgiSydeFZuPH8BCQCGifSrUvXtlSgGJ+cNyc4sfLEiGY3k/BaDm1TcPUwYJ6GlHErSBxTYsbxZAGPrEKI+52ArDxfSgVYklSKaIEi9PCLV5Gi9rGzkHbix//gj6ZvTgYdhmVI4IExCTRlqvgZzOtGRe5NH5iqizg9xq1oql+RrrD9bA3ZfUHePYNAtXX7KdjOLldYwY+gKLg7n+SadGXSEXbMHr6MZSrZvjmnUZ5SaNNfqeEHVcZFSBeeZPDBF/5AGDBhnLsIjqL19TMQio3164XW5TvtW6dDdpEarF9n7sgWDMvOkbOdBCb5/gxOnktj6KKpiamL7ce3OhE1eWsLTX7+snFX33Sf6uunYN/5vilbTLyxRJccpd3E8wmBH22tBKPys609haj4XMqelaRfji1lTnwKeEeb3+FJoN9V8KWLuj5d18WEPk33JrInOj03ynQ+ACoIHKD1+05iAWBNT2i5T7RRjUCmKJdO/iFP9rJHPOm59PsOB0l+oyb4s8bxYZn8qJWF37pNiiS5fMimg3Q/4Py7mAPJH8Qs4WTDQ4QMZ68FbaTodRfMxsUnNIZcFP8c3r5jJK5ZVYuwTPIrt2pSAdAXvnBMmSRD1y5S6bhpk9pZ8HqFHKhduDGJU3ErHrRpuzm9RR7TL11wOSSt88tORgIzDJIwkiuz4JWmfxkRTaeSCPo9/OqIlmcZTRkDkIhDU0F2wxMKv4QL1FU43Yt3ijYNGqYeQ9BjBLIA9Y9KAX5up8b4PBu9GoQlJVUeMNO47pdYFtsoF/BLxDXt3w76fxze/tfluZFB+PUP4GXx05aya/4DnHQ+m7xLv7k7OvH6aBzTcdlcqbce71eVS03r0zxPqVvzcyOpJJRLOHWxyYDc6XtaKMpRXIkKO7X/SSWduQpSft/zcdxeUl9Uz/EH5YMGHk2TIFDwi7kPf3WOOMof9qbw0kPvj1kYmeUxWZvENSQyYSj+mopVWauarSPCSmHoGFeccyyLl5SFU5gm0ddeKe2IW0TSktcguTqDJVTlQfyHxe3wsMolFmmtiASar2F9+eGiz/B+5c88DSjjsCrrRV4piZ62v7GUtgdedzVVb4d8YVX6nunsKrHqfp0WW0mGxzqH6yPyTVlRxY/Ql/qIqoQS64rWNaRPzRFrEReAjKLxHo1ldFRmvn3uXEsEtiqr+W11M2p3MUwWPVVLvzu8aYFSafg3lia9qMyoX8HSJrVHvXVUI31Yh7mKERSGuUV/2Rql/5+lSN3cH9Eh13E9MYr+4XDpgNfIlwozKQP/DTvQyVk=')).decode()
assert hashlib.sha256(patch.encode()).hexdigest()=='2497f00d26f076da74278c300d619e585657e04bf3c59dd21109580c7dedaf92'

def make_checker(name,patched=False):
    module=types.ModuleType(name)
    sys.modules[name]=module
    ns=module.__dict__
    ns.update({k:globals()[k] for k in ['re','pd','np','torch','labse','util','nli_tok','nli_model','device','CONTRA']})
    for src in [frozen['model_functions'],*frozen['base_rules'],frozen['base_checker_functions'],frozen['merged_normalisation'],frozen['merged_checker']]:
        exec(src,ns)
    if patched: exec(patch,ns)
    assert (ns['SIM_T'],ns['NLI_T'])==(0.4,0.65)
    assert ns['RULE_SOURCE_HASHES']==frozen['source_hashes']
    return ns
checker = make_checker('v9_frozen_holdout', True)['check_notice']
results = []
started = datetime.now(timezone.utc).isoformat()
print('Running frozen V9 on',device,flush=True)
for i, case in enumerate(inputs,1):
    frame = checker(case['chinese'],case['english'])
    results.append(dict(case_id=case['case_id'], warnings=json.loads(frame.to_json(orient='records',force_ascii=False))))
    if i%50==0:
        print(f'{i}/1000 complete',flush=True)

counts = {tier:Counter() for tier in ('overall','routine','difficult','edge')}
paths = Counter()
for case, result in zip(references,results,strict=True):
    assert case['case_id']==result['case_id']
    flagged = any(w['why'] for w in result['warnings'])
    truth = case['expected_status']=='issue'
    outcome = ('T' if truth==flagged else 'F')+('P' if flagged else 'N')
    counts['overall'][outcome]+=1
    counts[case['difficulty']][outcome]+=1
    paths.update(set(w.get('decision_path','unknown') for w in result['warnings']))
for tier,c in counts.items():
    print(tier,dict(c),'accuracy',(c['TP']+c['TN'])/sum(c.values()))
print('Decision paths:',dict(paths))
metadata = dict(dataset_sha256=hashlib.sha256(raw).hexdigest(),
                candidate_sha256=hashlib.sha256(patch.encode()).hexdigest(),
                started=started,completed=datetime.now(timezone.utc).isoformat(),device=device,
                counts=counts,paths=paths)
output = json.dumps(dict(metadata=metadata,results=results),ensure_ascii=False)
Path('v9_unseen_1000_results.json').write_text(output)
export = base64.b64encode(zlib.compress(output.encode(),9)).decode()
display(HTML('<details><summary>Complete frozen V9 results export</summary><textarea id="v9-unseen-export" rows="5">'+export+'</textarea></details>'))
print('FROZEN V9 BENCHMARK COMPLETE',flush=True)
